# Fase 2 — Conhecendo a base e escolhendo as variáveis

Este notebook faz duas coisas: mostra o que tem dentro da base recém-montada
e decide quais variáveis seguem para a etapa seguinte. Dele saem a
**Tabela 2** (estatísticas descritivas e percentual de zeros) e a
**Figura 2** (matriz de correlação) do artigo.

**Uma palavra que aparece o tempo todo: *feature*.** É a variável que vai ser
usada para comparar um município com o outro. Nem toda coluna da base é
feature: algumas são identificação (nome, código do IBGE), outras são
contexto guardado para consulta. Quem diz o papel de cada coluna é o
`dicionario_base.csv`.

**Por que 644 e não 645 municípios.** Trabalhamos só com os municípios que
têm nota do IEGM (Índice de Efetividade da Gestão Municipal), calculado pelo
TCE-SP (Tribunal de Contas do Estado de São Paulo). A capital fica de fora
porque quem a fiscaliza é outro órgão, o TCM-SP (Tribunal de Contas do
Município de São Paulo), e por isso ela não recebe IEGM. São Paulo continua
na base, só não entra nesta análise nem na modelagem.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Os notebooks ficam em notebooks/ e o código do projeto em src/. Estas duas
# linhas apontam o Python para src/, para os "from config import ..." abaixo
# funcionarem tanto rodando daqui quanto da raiz do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import (BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSED,
                    POPULACAO_MINIMA, LIMIAR_REDUNDANCIA)
from estilo import aplicar_estilo, salvar
from figuras import (plot_hist_populacao, plot_boxplot_taxas,
                     plot_distribuicoes_log1p, plot_spearman)

aplicar_estilo()   # mesma fonte, cores e grade em todas as figuras do artigo
# Se algum import acima falhar, quase sempre é o editor apontando para outra
# instalação do Python. O caminho impresso aqui é o que precisa ter as
# bibliotecas do requirements.txt.
print("Python:", sys.executable)

# O código do IBGE é identificador, não quantidade: lido como texto para não
# virar número em nenhuma etapa.
base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})
dic = pd.read_csv(DICIONARIO_CSV)
print(f"base_final: {base.shape[0]} municípios x {base.shape[1]} colunas")

Em vez de escrever a lista de colunas à mão, lemos tudo do
`dicionario_base.csv`, que o `merge_bases.py` gera junto com a base. Para
cada coluna ele diz de que **bloco** ela veio (criminalidade, socioeconômico
ou gestão) e qual é o **papel** dela. Os papéis que interessam aqui:

- `feature` - entra na comparação entre municípios;
- `feature_tier2` - é uma medida válida, mas frágil demais para comparar
  municípios: ou quase todos ficam no mesmo valor, ou ela só existe onde
  houve ocorrência. Continua na base, fora da modelagem;
- os demais (`chave`, `rotulo`, `contexto`, `metadado`...) ficam na base para
  consulta e para as tabelas do artigo, mas não entram em nenhuma conta.

Ler daqui, em vez de repetir os nomes das colunas no notebook, significa que
uma mudança de papel feita no `merge_bases.py` chega sozinha até aqui.

Um efeito disso aparece logo: vão ser **9** taxas criminais nesta análise, e
não as 10 que o `relatorio_base.txt` lista. O roubo de carga é
`feature_tier2`, porque 57% dos municípios não registraram nenhum caso em
três anos.

In [ ]:
# bloco_de["taxa_cvli"] devolve "criminalidade". Este atalho é usado o
# notebook inteiro para saber de que fonte cada coluna veio.
bloco_de = dic.set_index("coluna")["bloco"]
ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]
# Ordena as features por bloco (e por nome dentro do bloco), para as tabelas
# e a Figura 2 saírem sempre agrupadas na mesma ordem.
features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],
                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))
# As taxas criminais são reconhecidas pelo bloco, e não pelo nome começar com
# "taxa_": taxa_alfabetizacao e taxa_urbanizacao começam igual e são
# socioeconômicas.
taxas = [c for c in features if bloco_de[c] == "criminalidade"]
ordinais = [c for c in features if bloco_de[c] == "gestao"]

# flag_sem_iegm marca a capital; o "~" inverte a marcação e sobram os 644.
modelagem = base[~base["flag_sem_iegm"]].copy()
print(f"{len(features)} features em {len(modelagem)} municípios:")
print(pd.Series([bloco_de[c] for c in features]).value_counts().reindex(ORDEM_BLOCOS).to_string())

## 0. Primeiro contato com a base

Antes de transformar ou decidir qualquer coisa: o que tem na base, o que
está faltando e como os valores se espalham. Nenhuma decisão de modelagem é
tomada nesta seção. Ela existe para a gente conhecer o material com que vai
trabalhar, e é onde aparecem as perguntas que as seções seguintes respondem.

In [ ]:
# Quantas linhas, quais colunas, de que tipo e quantas estão preenchidas.
base.info()

In [ ]:
# Só as colunas que têm algum valor faltando.
ausentes = base.isna().sum()
ausentes[ausentes > 0]

Os ausentes são de dois tipos, e nenhum deles é erro de coleta.

As colunas do IEGM faltam em um município só, a capital, pelo motivo já
explicado. E as duas proporções de veículo faltam nos municípios que não
tiveram nenhum veículo roubado ou furtado em três anos: proporção é uma
divisão, e ali o denominador é zero.

Isso importa para a etapa seguinte: como nenhum ausente é aleatório, não faz
sentido preencher esses vazios com média ou mediana. Os municípios sem IEGM
saem da modelagem, e as duas proporções de veículo não são usadas como
feature.

In [ ]:
# describe() resume cada coluna: contagem, média, desvio padrão, mínimo,
# quartis e máximo. O .T vira a tabela de lado (uma linha por variável), que
# fica bem mais fácil de ler quando são muitas colunas.
for b in ORDEM_BLOCOS:
    cols = [c for c in features if bloco_de[c] == b]
    print(f"\n--- {b} ---")
    display(modelagem[cols].describe().round(2).T)

Três coisas para reparar nessas tabelas.

**Nas taxas criminais, a média é sempre maior que a mediana**, e o máximo
fica muito longe do terceiro quartil (o valor que separa os 75% menores do
resto). Esse é o desenho típico de uma distribuição de *cauda longa à
direita*: a maioria dos municípios com valores baixos e uns poucos bem acima
de todo mundo, que puxam a média para cima. Isso volta na seção 2, porque
atrapalha o cálculo de distância entre municípios.

**No bloco socioeconômico, as duas variáveis em reais medem coisas
diferentes.** O PIB per capita chega a R$ 580 mil por habitante, enquanto a
renda domiciliar mediana não passa de R$ 2.500. Não é contradição: o PIB per
capita é toda a produção do município dividida pelo número de moradores,
então uma refinaria ou uma usina joga o valor lá em cima sem que ninguém ali
ganhe mais por isso.

**No IEGM, as notas ficam na parte baixa da escala.** A escala vai de 1
(conceito C) a 5 (conceito A), e o valor de cada município é a média dos
exercícios de 2022 a 2024, daí os valores quebrados. As medianas das sete
dimensões ficam entre 1,0 e 2,7, ou seja, quase nenhum município chega perto
do topo. Notas empilhadas num canto só separam pouco, e esse ponto volta na
seção seguinte.

In [ ]:
# Escala log no eixo x: sem ela, São Paulo e as poucas cidades grandes
# esticam o gráfico e os 600 municípios pequenos viram uma barra só.
fig = plot_hist_populacao(base["populacao"], POPULACAO_MINIMA)
salvar(fig, "figura_eda_populacao")

Quase um quarto dos municípios paulistas (149 dos 645) tem menos de 5.000
habitantes, e isso é um problema conhecido de qualquer indicador em forma de
taxa. Numa cidade de 3.000 pessoas, uma única ocorrência em três anos já
vira uma taxa de 11 por 100 mil habitantes/ano. O número está certo, mas não
é comparável com os 11 por 100 mil de uma cidade de 300 mil habitantes: o
primeiro muda completamente se acontecer um caso a mais, o segundo não.

A base marca esses municípios com a coluna `flag_pop_pequena`. Eles não são
excluídos, porque excluir um quarto do estado seria mudar a pergunta do
trabalho. Ficam marcados para serem isolados depois, numa análise de
sensibilidade.

In [ ]:
# Boxplot: a caixa cobre os 50% do meio, a linha dentro dela é a mediana e os
# pontos soltos são os municípios muito acima do resto. O eixo y está em
# escala log para as nove taxas, que têm ordens de grandeza diferentes,
# caberem no mesmo gráfico.
fig = plot_boxplot_taxas(modelagem, taxas)
salvar(fig, "figura_eda_boxplot_taxas")

In [ ]:
# nlargest(10, coluna) devolve os dez municípios com o maior valor na coluna.
def top10(coluna):
    return (modelagem.nlargest(10, coluna)[["municipio", "populacao", coluna]]
            .round(1).reset_index(drop=True))

# As duas listas lado a lado, para dar para comparar quem aparece em cada uma
# (repare na coluna de população).
pd.concat([top10("taxa_cvli"), top10("taxa_roubo_outros")], axis=1,
          keys=["taxa_cvli", "taxa_roubo_outros"])

As duas listas não têm nenhum município em comum, e a coluna de população
explica por quê.

No topo do CVLI estão cidades pequenas: a população mediana desses dez é de
cerca de 7 mil habitantes. É exatamente o efeito descrito acima, com poucos
moradores, dois ou três casos em três anos bastam para liderar o ranking. O
valor é verdadeiro, mas instável: no período seguinte, sem nenhum caso, a
mesma cidade despenca para o fim da lista.

No topo do roubo estão cidades médias e grandes, com população mediana perto
de 317 mil habitantes, onde o volume é alto o suficiente para a taxa ser
estável.

Ou seja, as duas taxas não medem a mesma coisa nem se comportam do mesmo
jeito. É um primeiro argumento a favor de levar as taxas separadas para a
modelagem, em vez de somar tudo num índice único de criminalidade.

In [ ]:
# Três jeitos de ver se uma nota separa ou não os municípios: qual valor mais
# se repete, que fatia dos municípios está nele e quantos valores diferentes
# a coluna assume.
resumo_iegm = pd.DataFrame({
    "valor mais comum": modelagem[ordinais].mode().iloc[0],
    "% no valor mais comum": (modelagem[ordinais]
                              .apply(lambda s: s.value_counts(normalize=True).max()) * 100).round(0),
    "valores distintos": modelagem[ordinais].nunique(),
})
resumo_iegm

As notas do IEGM variam pouco. O caso extremo é o `i_planejamento_ord`: 70%
dos municípios receberam exatamente a mesma nota. Uma variável em que a
maioria tem o mesmo valor quase não ajuda a diferenciar um município do
outro.

## 1. Tabela 2 - descritivas das taxas e percentual de zeros

**`% zeros`.** Um zero numa taxa criminal não quer dizer "pouco crime",
quer dizer "nenhuma ocorrência registrada em três anos". Uma variável com
30% de zeros está misturando dois tipos de município na mesma coluna: os que
têm o fenômeno em grau baixo e os que não têm o fenômeno. Nenhuma
transformação matemática desfaz essa mistura, então o que dá para fazer é
saber o tamanho dela desde já.

**`assimetria`.** Mede o quanto a distribuição pende para um lado. Zero
significa simétrica; positiva significa cauda longa à direita, que é o caso
de todas as taxas aqui. Quanto maior o valor, mais a distribuição é formada
por muitos municípios baixos e poucos muito altos.

In [ ]:
tabela2 = pd.DataFrame({
    "média": modelagem[taxas].mean(),
    "mediana": modelagem[taxas].median(),
    "desvio": modelagem[taxas].std(),
    "máx": modelagem[taxas].max(),
    "% zeros": (modelagem[taxas] == 0).mean() * 100,
    # skew() é a assimetria: 0 = simétrica, positiva = cauda longa à direita.
    "assimetria": modelagem[taxas].skew(),
}).round(2)
# Fica salva em disco porque é uma tabela do artigo, não um resultado de
# passagem.
tabela2.to_csv(DATA_PROCESSED / "tabela2_descritivas.csv")
tabela2

## 2. As distribuições antes e depois do `log1p`

A cauda longa atrapalha qualquer conta baseada em distância entre
municípios, e distância é exatamente o que a clusterização vai usar. Com uma
coluna indo de 0 a 950 e outra de 0 a 100, a primeira decide quase sozinha
quem é parecido com quem, não por ser mais importante, e sim por estar numa
escala maior.

O usual é o logaritmo: ele encolhe bastante os valores grandes e
quase nada os pequenos, o que traz a cauda de volta para perto do corpo da
distribuição. Usamos o `log1p`, que calcula `log(1 + x)`. O `+ 1` está ali
porque o logaritmo de zero é indefinido, e as nossas taxas têm muitos zeros;
somando 1 antes, o zero vira zero e nenhum dado precisa ser inventado nem
descartado.

Aplicamos nas 9 taxas criminais e nas 2 variáveis em reais (PIB per capita e
renda mediana), que são as de cauda longa. Percentuais, que já vão de 0 a
100, e notas de 1 a 5 não precisam.

In [ ]:
monetarias = ["pib_percapita", "renda_domiciliar_mediana"]
log_cols = taxas + monetarias

# Cada variável aparece duas vezes: à esquerda como está na base, à direita
# depois do log1p, com a assimetria no título de cada uma.
fig = plot_distribuicoes_log1p(modelagem, log_cols)
salvar(fig, "figura_distribuicoes_log1p")

Repare que, depois do `log1p`, várias taxas ficam com assimetria
**negativa**, a cauda passou a apontar para a esquerda. Isso não quer dizer
que a transformação exagerou na dose.

O que está acontecendo é o monte de zeros. O `log1p` deixa o zero em zero,
enquanto todos os outros valores vão parar perto de 3 ou 4. O resultado é um
pico isolado na ponta esquerda do gráfico, longe do resto, e é ele que puxa
a assimetria para o negativo. Nem é preciso ter muitos zeros: o estupro tem
só 1% deles e mesmo assim fica em −2,3, porque aqueles poucos municípios no
zero estão a quase quatro unidades de distância de todo o resto.

A tabela abaixo testa essa explicação. Se a causa forem mesmo os zeros,
calcular a assimetria só entre os municípios com valor maior que zero deve
devolver um número perto de zero.

In [ ]:
pd.DataFrame({
    "% zeros": (modelagem[taxas] == 0).mean() * 100,
    "assimetria bruta": modelagem[taxas].skew(),
    "assimetria log1p": np.log1p(modelagem[taxas]).skew(),
    # A mesma conta da coluna anterior, mas só com quem tem valor > 0: é o
    # teste de que o negativo vem dos zeros, e não da transformação.
    "assimetria log1p sem zeros": [np.log1p(modelagem.loc[modelagem[c] > 0, c]).skew()
                                   for c in taxas],
}).round(2)

Confirmado: sem os zeros, a assimetria não passa de 0,7 em módulo em nenhuma
das nove taxas, contra os +1,1 a +3,9 que elas tinham antes de qualquer
transformação. O `log1p` está fazendo o que se espera dele, e fica mantido.

Os zeros continuam sendo um problema, mas de outra natureza, que
transformação nenhuma resolve, eles são ausência do fenômeno, não escala
errada. Vão ser tratados na análise de sensibilidade, separando os
municípios marcados com `flag_pop_pequena`.

## 3. Figura 2 - a matriz de correlação

Correlação mede se duas variáveis andam juntas: perto de +1 quando uma sobe
e a outra também, perto de −1 quando uma sobe e a outra desce, perto de 0
quando não há relação. Aqui ela serve para responder se alguma dupla de
features está medindo a mesma coisa duas vezes.

Usamos a correlação de **Spearman**, e não a de Pearson, que é a mais
conhecida. A diferença: Pearson trabalha com os valores e enxerga só relação
em linha reta; Spearman troca cada valor pela posição dele no ranking e
enxerga qualquer relação que seja sempre crescente ou sempre decrescente.
Como as nossas taxas têm cauda longa, trabalhar com posições é mais seguro,
um município com taxa fora da curva vira "o primeiro colocado" em vez de um
número que distorce a conta inteira.

Na figura, as variáveis aparecem agrupadas por bloco, com linhas cinza
separando os três. Isso permite ler a figura por quadrantes: correlação alta
dentro de um bloco é esperada; entre blocos diferentes é o que realmente
interessa. Só os pares com |ρ| ≥ 0,5 recebem o número escrito, para a figura
não virar uma parede de dígitos.

In [ ]:
# corr() devolve a matriz de todos contra todos: 22 x 22.
rho = modelagem[features].corr(method="spearman")

fig = plot_spearman(rho, bloco_de, ORDEM_BLOCOS)
salvar(fig, "figura2_spearman")

## 4. Poda por redundância

Agora a regra objetiva. Se duas features têm |ρ| acima de
`LIMIAR_REDUNDANCIA` (0,85, definido no `config.py`), elas estão medindo
praticamente a mesma coisa. Manter as duas daria peso dobrado a esse aspecto
na hora de comparar municípios, então uma delas sairia.

O limiar é uma escolha nossa, não uma regra universal, por isso fica no
`config.py`, num lugar fácil de achar e de justificar no artigo, em vez de
espalhado pelos notebooks.

In [ ]:
# Percorre cada par de features uma vez só: o features[i + 1:] evita comparar
# a coluna com ela mesma e evita repetir o mesmo par ao contrário.
pares = [(a, b, round(rho.loc[a, b], 3))
         for i, a in enumerate(features) for b in features[i + 1:]
         if abs(rho.loc[a, b]) > LIMIAR_REDUNDANCIA]

if pares:
    display(pd.DataFrame(pares, columns=["a", "b", "rho"]))
else:
    print(f"Nenhum par com |rho| > {LIMIAR_REDUNDANCIA}: as {len(features)} "
          "features seguem para o pré-processamento.")

# Mesmo sem nenhum par passar do limiar, vale ver quais chegaram mais perto.
# A matriz é simétrica, então olhamos só o triângulo de baixo (tril), com
# k=-1 para tirar a diagonal, que é sempre 1. O stack() transforma a matriz
# numa lista de pares, que dá para ordenar.
maiores = rho.where(np.tril(np.ones_like(rho, dtype=bool), k=-1)).stack()
maiores.abs().sort_values(ascending=False).head(8).round(3)

## 5. Conclusões

1. **Nenhuma feature é descartada.** O par mais correlacionado de toda a
   matriz é `roubo_outros × roubo_veiculo`, com ρ = 0,76, bem abaixo do
   limiar de 0,85. Logo atrás vem a urbanização, com a coleta de lixo (0,74)
   e com o esgoto (0,67) — o que faz sentido, já que os três descrevem
   infraestrutura urbana, mas também não chega ao limiar. As 22 features
   seguem inteiras para o notebook 03.
2. **O quadrante criminalidade × gestão é quase todo branco.** A correlação
   mais forte entre um indicador de crime e uma nota de gestão é 0,37, e a
   mediana do quadrante é 0,10. Ou seja, saber a nota de gestão de um
   município não diz quase nada sobre a criminalidade dele, e vice-versa.
   Esse resultado é o que sustenta a proposta do trabalho: se os dois blocos
   dissessem a mesma coisa, juntá-los não acrescentaria informação nenhuma.
3. O `log1p` nas taxas e nas variáveis em reais fica confirmado, e os zeros
   ficam registrados como limitação a ser tratada depois.
4. Saídas geradas: `data/processed/tabela2_descritivas.csv` e
   `figuras/figura2_spearman.png`.